In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk
from mphsweepkit.plot_fields import PlotField

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('core_cross_section_solved.mph')

Load an already solved CascadedSweepModel

In [19]:
csm = msk.CascadedSweepModel(model, 'Study on Cross-Sections', show_param_names=True)

Initialized CascadedSweepModel
Study name: Study on Cross-Sections
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'hor_slit', 'vert_slit', 'w', 'l_r', 'a_e'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.core'
        - Excitation Sweep (Parametric) -> 'b_mean'
          - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Frequency Sweep']
Loop lengths: [3, 1, 4, 13]
--------------------------------
Data updated from MPh-model.
Input data shape: (156, 8)
Reset output data to shape of the input data: (156, 0)
Combined shape: (156, 8)


In [20]:
# 1) 
csm.print_available_selections()
print("\n")

# 2)
# the selection name must be chosen as defined in the COMSOL model.
selection_name = "Core"
selection_type = "dom"
selection_dataset_name = "Solution Core"

# 3)
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type="dom")

print(f"Selected domain tag: {selection_domain_tag}")
print("\n")

# 4)
# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)
# remove a dataset:
# csm.node_datasets.children()[-1].remove()

Available selections:
geom1_csel1_pnt Core
geom1_csel1_bnd Core
geom1_csel1_dom Core
geom1_csel2_pnt Air
geom1_csel2_bnd Air
geom1_csel2_dom Air
geom1_csel4_pnt Vertical Slit
geom1_csel4_bnd Vertical Slit
geom1_csel4_dom Vertical Slit
geom1_csel5_pnt Horizontal Slit
geom1_csel5_bnd Horizontal Slit
geom1_csel5_dom Horizontal Slit
geom1_csel3_pnt Region
geom1_csel3_bnd Region
geom1_csel3_dom Region


Selected domain tag: geom1_csel1_dom


Dataset 'Solution Core' already exists.
Available datasets: ['Study on Cross-Sections//Solution 1', 'Cascaded Sweep Solution', 'Solution Core']


In [21]:
# 5)
# Evaluate field expressions on the selection dataset and export the results to text files.
# For each geometry configuration, a separate text file is created with the corresponding results.
# This allows mesh reuse for expression evaluation and straightforward result comparison.

comsol_looplevels = csm.get_comsol_looplevels()

# Load customized post-processing expressions
post_processing_exprs = msk.load_post_processing_exprs("fields_post_processing_expressions.json",
                                                       print_info=True)

# Convert the post-processing expressions into lists for the COMSOL export
expressions = [entry["expression"] for entry in post_processing_exprs.values()]
labels = [entry["label"] for entry in post_processing_exprs.values()]
units = [entry["unit"] for entry in post_processing_exprs.values()]


# TODO: Add a visualization of the different geometries

for looplevel_arr in comsol_looplevels:

    export_node = csm.export_dataset_with_expressions(
        dataset_name=selection_dataset_name,
        expressions=expressions,
        descriptions=labels,
        units=units,
        looplevel=looplevel_arr,
        looplevelinput=['manual'] * len(looplevel_arr)
    )

    print(f"Loop level: {export_node.property('looplevel')}\n")

Loading post-processing expressions from: X:\Till_data\repositories\MPhSweepKit\examples\studies\infinite_ferrite_cross_sections\fields_post_processing_expressions.json
Loaded post-processing expressions:
{
  "norm_b": {
    "expression": "mf.normB",
    "unit": "T",
    "label": "Magnetic Flux Density"
  },
  "norm_e": {
    "expression": "mf.normE",
    "unit": "V/m",
    "label": "Electric Field"
  }
}
Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
{'allowdescrupdate': True, 'alwaysask': False, 'anythingwritten': True, 'batch': False, 'clearfilenameafterwards': False, 'compactexprs': True, 'config': 'none', 'const': [[]], 'constUnit': [], 'constactive': 'off', 'constisupdated': True, 'constprefixes': [''], 'coordfilename': WindowsPath('.'), 'data': 'dset3', 'datasupportsgp': 'on', 'descr': ['Magnetic Flux Density', 'Electric Field'], 'differential': Fa

Release the Client

In [22]:
client.remove(model)
client.clear()
client.disconnect()